In [ ]:
# Install dependencies first: python -m pip install -r requirements.txt

In [ ]:
# Step 1: Import necessary libraries
import praw
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# This version uses relative file paths for local execution.


In [ ]:
# Step 2: Set up a connection to Reddit API using My Client ID and Client Secret
import os
client_id = os.environ.get("REDDIT_CLIENT_ID")
client_secret = os.environ.get("REDDIT_CLIENT_SECRET")
if not client_id or not client_secret:
    raise RuntimeError("Set REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET as environment variables before running.")
reddit = praw.Reddit(client_id=client_id,
                     client_secret=client_secret,
                     user_agent=os.environ.get("REDDIT_USER_AGENT", "data-collection-portfolio/1.0"))

In [ ]:
# Step 3: Data Collection
# Select the subreddit we want to collect data from.
subreddit = reddit.subreddit('DataScience') # Here we used 'DataScience'

# List to store comment data
comments_data = []

# Collect 50 comments from posts in the subreddit
for submission in subreddit.hot(limit=50):  # You can replace 'hot' with 'top' or 'new'
    submission.comments.replace_more(limit=0)  # Remove more comments to load comments directly
    for comment in submission.comments.list():
        comments_data.append([comment.author, comment.body, comment.score, comment.created_utc, submission.title])

# Convert data to DataFrame
df_comments = pd.DataFrame(comments_data, columns=['Author', 'Comment', 'Score', 'Created UTC', 'Post Title'])

# Display initial rows of the dataset
print(df_comments.head())

# Export data to CSV file before processing

# Export data to CSV file in Google Drive
df_comments.to_csv('reddit_comments_data.csv', index=False)

# Export data to CSV file in my dvice
df_comments.to_csv('reddit_comments_data.csv', index=False)

# **# OR**

In [ ]:
# Step 3: Data Collection
# Select the subreddit you want to collect data from.
subreddit = reddit.subreddit('programming')  # Here we used 'programming'

# List to store posts data
posts_data = []

# Collect 750 posts from the subreddit
for submission in subreddit.hot(limit=750):  # You can replace 'hot' with 'top' or 'new'
    # Extract various information from posts
    # Text processing and blank columns
    text = submission.selftext if submission.selftext else "No text content"
    nsfw = submission.over_18  # Will give True if the content is NSFW, False if it is not.
    flair = submission.link_flair_text if submission.link_flair_text else "No flair"

    # Extract various information from posts
    posts_data.append([
        submission.title,             # Title
        submission.score,             # Rating
        submission.id,                # Post ID
        submission.url,               # Post Link
        submission.num_comments,      # Number of Comments
        text,                         # Text (or alternate value if not present)
        submission.created_utc,       # Creation Date (in UTC format)
        submission.author,            # Author Name
        submission.subreddit.display_name,  # subreddit name
        nsfw,                         # Does it contain NSFW content?
        flair                         # The text for the flair (or the alternative value if it does not exist)
    ])

# Convert data to DataFrame
df_posts = pd.DataFrame(posts_data, columns=[
    'Title', 'Score', 'ID', 'URL', 'Comments Count', 'Text',
    'Created (UTC)', 'Author', 'Subreddit', 'NSFW', 'Flair'
])

# Display initial rows of the dataset
#print(df_posts.head())

# Export data to CSV file before processing

# Export data to CSV file in Google Drive
df_posts.to_csv('reddit_programming_posts_data.csv', index=False)

# Export data to CSV file in my dvice
df_posts.to_csv('reddit_programming_posts_data.csv', index=False)

In [ ]:
# Step 4: Data Exploration
print(df_posts.head())  # Show first 5 records
print('*'*100)
print(df_posts.info())  # Display basic information about the dataset
print('*'*100)
print(df_posts.describe())  # Summary statistics for numerical features

In [ ]:
# Display column names and types for feature descriptions

print("Columns in the dataset:")
print(df_posts.columns)

In [ ]:
# Step 5: Data Cleaning

# Check for missing values
print("Missing values in the dataset:")
print(df_posts.isnull().sum()) # Find out which columns contain empty values

In [ ]:
# Handle missing values
df_posts['Text'].fillna('No text content', inplace=True)
df_posts['Flair'].fillna('No flair', inplace=True)

# Handle missing values with mean for numerical columns
df_posts['Comments Count'].fillna(df_posts['Comments Count'].mean(), inplace=True)
print(df_posts.isnull().sum())


In [ ]:
# Check for duplicated values
print("duplicated values in the dataset:")
print(df_posts.duplicated().sum())

In [ ]:
# Handle duplicated values
df_posts.drop_duplicates(inplace=True)
#data.drop_duplicates()

print(df_posts.duplicated().sum())# Ensure that there are no empty values ​​after processing.

In [ ]:
# Step 6: Data Transformation

from sklearn.preprocessing import StandardScaler, LabelEncoder

# Normalize the 'Score' column using StandardScaler
scaler = StandardScaler()
df_posts['Normalized Score'] = scaler.fit_transform(df_posts[['Score']])

# Encoding categorical columns (e.g. 'NSFW' and 'Flair')
label_encoder = LabelEncoder()
df_posts['NSFW'] = label_encoder.fit_transform(df_posts['NSFW'])
df_posts['Flair'] = label_encoder.fit_transform(df_posts['Flair'])


In [ ]:
# Display data after conversions
print(df_posts.head())

In [ ]:
# Display column names and types for feature descriptions

print("Columns in the dataset:")
print(df_posts.columns)


In [ ]:
df_posts.info()
print('='*100)
print(df_posts.isnull().sum())
print('='*100)

In [ ]:
# Step 7: Feature Engineering

# Create new feature: Text Length (number of characters in text)
df_posts['Text Length'] = df_posts['Text'].apply(len)

# Convert creation date (UTC) to date format
df_posts['Created Date'] = pd.to_datetime(df_posts['Created (UTC)'], unit='s')

# Create new feature: Year of creation
df_posts['Year Created'] = df_posts['Created Date'].dt.year

In [ ]:
# Display column names and types for feature descriptions

print("Columns in the dataset:")
print(df_posts.columns)
print('='*100)
df_posts.info()
print('='*100)
print(df_posts.isnull().sum())
print('='*100)

In [ ]:
# Additional step => Data Exploration (visualizations.)
df_posts.hist()

In [ ]:
# Create a boxplot to distribute ratings
df_posts.boxplot()

In [ ]:
# Treatment Summary
print("Data Collection: 750 posts collected from the 'programming' subreddit.")
print("Data Exploration: Some initial statistics and columns are shown.")
print("Data Cleaning: Empty values ​​are processed, duplicates are removed.")
print("Data Transformation: The 'Score' column is normalized and the categorical columns are coded.")
print("Feature Engineering: New features like 'Text Length' and 'Year Created' have been added.")


In [ ]:
# Step 8: Summary of Processing
print("Data Collection: 750 posts collected from the 'programming' subreddit.")
print("Data Exploration: Some initial statistics and columns are shown.")
print("Data Cleaning: Empty values ​​are processed, duplicates are removed.")
print("Data Transformation: The 'Score' column is normalized and the categorical columns are coded.")
print("Feature Engineering: New features like 'Text Length' and 'Year Created' have been added.")

# Display the final transformed dataset
#df_posts.head()
print('='*80)
print("Programing by: Mahmoud M. M. Ashour")
print("*****Assignment 1: Data Collection and Preparation(Dataset 3 with API)*****")
print("*****Advanced Data Science*****")
print("*****Dr. Iyad Husni Alshami*****")
print('='*80)

In [ ]:
# Export data to CSV file after processing

# Export data to CSV file in Google Drive
df_posts.to_csv('processed_reddit_programming_posts_data.csv', index=False)

# Export data to CSV file in my dvice
df_posts.to_csv('processed_reddit_programming_posts_data.csv', index=False)
